# 24.6 Particle swarm optimization

A swarm moves by inertia, personal memory, and social memory of the best discovered position.

Particle swarm optimization is population search written as motion. It is useful when candidate solutions are continuous and the geometry is easier to explore by moving points than by recombining genomes.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 2406
rng = np.random.default_rng(SEED)


def sphere_loss(x):
    x = np.asarray(x, dtype=float)
    return float(np.sum(x * x))


def rastrigin_loss(x):
    x = np.asarray(x, dtype=float)
    n = x.size
    waves = x * x - 10.0 * np.cos(2.0 * np.pi * x)
    return float(10.0 * n + np.sum(waves))


def constrained_loss(x):
    x = np.asarray(x, dtype=float)
    base = sphere_loss(x - np.array([1.0, -1.0]))
    violation = max(0.0, x[0] + x[1] - 1.0)
    lower = np.maximum(0.0, -5.0 - x)
    upper = np.maximum(0.0, x - 5.0)
    bounds_penalty = float(np.sum(lower * lower + upper * upper))
    return float(base + 50.0 * violation * violation + 20.0 * bounds_penalty)


def make_black_box_ladder():
    ladder = []
    ladder.append({
        "name": "D1 1-D quadratic",
        "dim": 1,
        "bounds": np.array([[-5.0, 5.0]]),
        "objective": lambda x: float((np.asarray(x)[0] - 3.0) ** 2),
        "optimum": np.array([3.0]),
    })
    ladder.append({
        "name": "D2 2-D sphere",
        "dim": 2,
        "bounds": np.array([[-5.0, 5.0], [-5.0, 5.0]]),
        "objective": sphere_loss,
        "optimum": np.zeros(2),
    })
    ladder.append({
        "name": "D3 multimodal Rastrigin",
        "dim": 2,
        "bounds": np.array([[-5.12, 5.12], [-5.12, 5.12]]),
        "objective": rastrigin_loss,
        "optimum": np.zeros(2),
    })
    ladder.append({
        "name": "D4 constrained penalty",
        "dim": 2,
        "bounds": np.array([[-5.0, 5.0], [-5.0, 5.0]]),
        "objective": constrained_loss,
        "optimum": np.array([1.0, -1.0]),
    })
    ladder.append({
        "name": "D5 high-dim Rastrigin",
        "dim": 30,
        "bounds": np.tile(np.array([[-5.12, 5.12]]), (30, 1)),
        "objective": rastrigin_loss,
        "optimum": np.zeros(30),
    })
    return ladder


def sample_uniform(bounds, count, generator):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    samples = generator.uniform(lows, highs, size=(count, len(bounds)))
    return samples


def clip_to_bounds(x, bounds):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    return np.minimum(np.maximum(x, lows), highs)


def reflect_to_bounds(x, v, bounds):
    lows = bounds[:, 0]
    highs = bounds[:, 1]
    below = x < lows
    above = x > highs
    x = np.minimum(np.maximum(x, lows), highs)
    v = np.where(below | above, -0.5 * v, v)
    return x, v


def plot_landscape(ax, rung):
    dim = rung["dim"]
    bounds = rung["bounds"]
    if dim == 1:
        xs = np.linspace(bounds[0, 0], bounds[0, 1], 200)
        ys = np.array([rung["objective"]([x]) for x in xs])
        ax.plot(xs, ys, color="0.35")
        ax.set_xlabel("x")
        ax.set_ylabel("loss")
        return
    grid_x = np.linspace(bounds[0, 0], bounds[0, 1], 80)
    grid_y = np.linspace(bounds[1, 0], bounds[1, 1], 80)
    xx, yy = np.meshgrid(grid_x, grid_y)
    zz = np.zeros_like(xx)
    for row in range(xx.shape[0]):
        for col in range(xx.shape[1]):
            probe = np.zeros(dim)
            probe[0] = xx[row, col]
            probe[1] = yy[row, col]
            zz[row, col] = rung["objective"](probe)
    ax.contourf(xx, yy, zz, levels=24, cmap="viridis")
    ax.set_xlabel("x0")
    ax.set_ylabel("x1")


## The concept, built once: the PSO update

PSO uses the lesson formula $$v_i\leftarrow wv_i+c_1r_1(p_i-x_i)+c_2r_2(g-x_i),\qquad x_i\leftarrow x_i+v_i.$$ We first check the hand update so the implementation is anchored to the lesson numbers.

In [ ]:
x = np.array([0.0, 4.0])
v = np.array([0.0, 0.0])
p = x.copy()
g = np.array([0.0, 0.0])
w = 0.5
c1 = 1.0
c2 = 1.0
r1 = np.array([0.2, 0.2])
r2 = np.array([0.3, 0.3])
new_v = w * v + c1 * r1 * (p - x) + c2 * r2 * (g - x)
new_x = x + new_v
assert np.allclose(new_v, [0.0, -1.2])
assert np.allclose(new_x, [0.0, 2.8])
speeds = []
for inertia in [0.2, 0.7, 1.0]:
    speed = np.linalg.norm(inertia * np.array([1.0, -0.5]))
    speeds.append(round(float(speed), 3))
assert speeds == [0.224, 0.783, 1.118]
print("new position", new_x)
print("inertia-only speeds", speeds)

Now we package the same motion rule into a reusable optimizer with personal bests, global or ring social memory, velocity clipping, and reflection at bounds.

In [ ]:

def particle_swarm_optimize(objective, bounds, n_particles=25, generations=45, w=0.7, c1=1.4, c2=1.4, vmax_frac=0.25, topology="global", seed=0):
    generator = np.random.default_rng(seed)
    dim = bounds.shape[0]
    positions = sample_uniform(bounds, n_particles, generator)
    span = bounds[:, 1] - bounds[:, 0]
    velocities = generator.uniform(-0.1 * span, 0.1 * span, size=(n_particles, dim))
    personal_best = positions.copy()
    personal_loss = np.array([objective(x) for x in positions])
    best_index = int(np.argmin(personal_loss))
    global_best = personal_best[best_index].copy()
    global_loss = float(personal_loss[best_index])
    history = [global_loss]
    traces = [positions.copy()]
    vmax = vmax_frac * span
    for generation in range(generations):
        for i in range(n_particles):
            if topology == "ring":
                neighbors = [(i - 1) % n_particles, i, (i + 1) % n_particles]
                local_index = neighbors[int(np.argmin(personal_loss[neighbors]))]
                social_best = personal_best[local_index]
            else:
                social_best = global_best
            r1 = generator.random(dim)
            r2 = generator.random(dim)
            inertia = w * velocities[i]
            cognitive = c1 * r1 * (personal_best[i] - positions[i])
            social = c2 * r2 * (social_best - positions[i])
            velocities[i] = inertia + cognitive + social
            velocities[i] = np.minimum(np.maximum(velocities[i], -vmax), vmax)
            positions[i] = positions[i] + velocities[i]
            positions[i], velocities[i] = reflect_to_bounds(positions[i], velocities[i], bounds)
        losses = np.array([objective(x) for x in positions])
        improved = losses < personal_loss
        personal_best[improved] = positions[improved]
        personal_loss[improved] = losses[improved]
        best_index = int(np.argmin(personal_loss))
        if personal_loss[best_index] < global_loss:
            global_loss = float(personal_loss[best_index])
            global_best = personal_best[best_index].copy()
        history.append(global_loss)
        traces.append(positions.copy())
    return {
        "best_x": global_best,
        "best_loss": global_loss,
        "history": np.array(history),
        "positions": positions,
        "traces": traces,
        "personal_loss": personal_loss,
    }


## The dataset ladder

We use the F4 black-box objective ladder inline: D1 is 1-D, D2 is a sphere, D3 is multimodal, D4 adds a constraint penalty, and D5 is high-dimensional.

In [ ]:
ladder = make_black_box_ladder()
for rung in ladder:
    preview = sample_uniform(rung["bounds"], 3, rng)
    values = [rung["objective"](row) for row in preview]
    print(rung["name"], "dim=", rung["dim"], "bounds_shape=", rung["bounds"].shape)
    print("sample", np.round(preview[:2], 3))
    print("loss", np.round(values[:2], 3))

In [ ]:
results = []
for idx, rung in enumerate(ladder):
    result = particle_swarm_optimize(rung["objective"], rung["bounds"], seed=SEED + idx)
    results.append(result)
    print(rung["name"], "best_loss=", round(result["best_loss"], 4), "best_fitness=", round(-result["best_loss"], 4), "convergence_gen=", int(np.argmin(result["history"])))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for idx, rung in enumerate(ladder):
    ax = axes[0, idx]
    plot_landscape(ax, rung)
    pts = results[idx]["positions"]
    if rung["dim"] == 1:
        yvals = [rung["objective"]([value]) for value in pts[:, 0]]
        ax.scatter(pts[:, 0], yvals, s=18, color="tab:red")
    else:
        ax.scatter(pts[:, 0], pts[:, 1], s=18, color="tab:red")
    ax.set_title(rung["name"])
summary_ax = axes[1, 0]
for idx, result in enumerate(results):
    summary_ax.plot(-result["history"], label=f"D{idx + 1}")
summary_ax.set_title("best fitness vs generation")
summary_ax.set_xlabel("generation")
summary_ax.set_ylabel("best fitness")
summary_ax.legend()
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on D5: social weight too high

A large social pull can collapse the swarm around an early lucky global best. The fix balances inertia, cognitive, and social terms, clips velocity, and uses local ring neighborhoods.

In [ ]:
d5 = ladder[-1]
bad = particle_swarm_optimize(d5["objective"], d5["bounds"], c1=0.2, c2=3.0, vmax_frac=1.0, topology="global", seed=99)
good = particle_swarm_optimize(d5["objective"], d5["bounds"], c1=1.4, c2=1.4, vmax_frac=0.18, topology="ring", seed=99)
bad_diversity = float(np.mean(np.std(bad["positions"], axis=0)))
good_diversity = float(np.mean(np.std(good["positions"], axis=0)))
print("bad best fitness", round(-bad["best_loss"], 3), "diversity", round(bad_diversity, 3))
print("fixed best fitness", round(-good["best_loss"], 3), "diversity", round(good_diversity, 3))

## Evaluate it + Practice

- Metric: best fitness and the generation where the best-so-far curve first reaches that value; compare with random search using the same objective-call budget.
- Sanity check: D1 should move toward the known optimum near 3.
- Ablation: set `c1=0` or `c2=0` and watch memory loss hurt convergence.
- Failure signals: all particles share nearly identical coordinates, or the best curve is flat early on D5.

Practice: change the ring neighborhood size and compare D5 diversity.

Practice: add a noisy objective wrapper and re-evaluate the global-best pitfall.